# Solar Energy Siting Suitability Analysis — Colorado

## Overview
This notebook implements a weighted multi-criteria suitability model to identify
high-potential areas for solar energy deployment across Colorado.

**Datasets:**
- SOLARGIS Solar resource maps & GIS data — Global Horizontal Irradiance (GHI)
- USGS PAD-US 3.0 — Protected areas exclusion zones
- US Census TIGER — Colorado state boundary

**Method:** Raster normalization → weighted overlay → exclusion masking
**CRS:** EPSG:32613 (UTM Zone 13N)
**Resolution:** 1 km

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
from data_prep import reproject_raster, clip_raster_to_boundary
from suitability import compute_suitability, save_suitability, load_raster
from visualize import plot_suitability_map

%matplotlib inline
print("All imports successful")

## Step 1: Data Inspection

Before any processing, inspect the raw inputs to understand CRS,
resolution, extent, and nodata values.

In [ ]:
# Inspect the raw GHI raster
with rasterio.open("../data/raw/GHI_2022_Colorado.tif") as src:
    print("=== Raw GHI Raster ===")
    print(f"CRS:        {src.crs}")
    print(f"Resolution: {src.res}")
    print(f"Shape:      {src.width} x {src.height} pixels")
    print(f"Bounds:     {src.bounds}")
    print(f"NoData:     {src.nodata}")
    print(f"Data type:  {src.dtypes[0]}")

## Step 2: Reprojection and Clipping

Reproject from geographic CRS (EPSG:4326) to UTM Zone 13N (EPSG:32613)
for accurate area calculations in meters, then clip to Colorado boundary.

In [ ]:
from pathlib import Path

PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

# Reproject
reproject_raster(
    "../data/raw/GHI_2022_Colorado.tif",
    str(PROCESSED / "ghi_utm13n.tif")
)

# Clip
clip_raster_to_boundary(
    str(PROCESSED / "ghi_utm13n.tif"),
    "../data/raw/colorado_boundary.shp",
    str(PROCESSED / "ghi_clipped.tif")
)

# Verify output
with rasterio.open(PROCESSED / "ghi_clipped.tif") as src:
    print(f"\nClipped raster CRS: {src.crs}")
    print(f"Shape: {src.width} x {src.height}")

## Step 3: Exclusion Masking and Suitability Scoring

Protected areas (national parks, wilderness areas, water bodies) are
applied as hard exclusion zones. Remaining areas are scored 0–1.

In [ ]:
suitability, transform, crs, meta = compute_suitability(
    ghi_path=str(PROCESSED / "ghi_clipped.tif"),
    exclusion_path="../data/raw/PADUS3_0.gpkg",
)

# Show score distribution
valid_scores = suitability[~np.isnan(suitability)].flatten()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(valid_scores, bins=50, color="#E8593C", edgecolor="white", alpha=0.85)
ax.set_xlabel("Suitability Score (0–1)")
ax.set_ylabel("Pixel Count")
ax.set_title("Distribution of Suitability Scores — Colorado")
ax.axvline(valid_scores.mean(), color="navy", linestyle="--",
           label=f"Mean: {valid_scores.mean():.3f}")
ax.legend()
plt.tight_layout()
plt.show()

## Step 4: Results — Suitability Map

In [ ]:
save_suitability(suitability, transform, crs, meta,
                 str(PROCESSED / "suitability.tif"))

fig, ax = plot_suitability_map(
    str(PROCESSED / "suitability.tif"),
    "../data/raw/colorado_boundary.shp",
    "../data/outputs/solar_suitability_colorado.png"
)
plt.show()

## Key Findings

- Highest suitability concentrated in **southeastern Colorado plains** —
  high irradiance, flat terrain, minimal protected areas
- Rocky Mountain corridor correctly excluded due to steep slopes
- National parks and wilderness areas (RMNP, Mesa Verde, Great Sand Dunes)
  visible as exclusion zones

## Limitations and Next Steps

- Slope layer not yet integrated (currently 100% GHI weight)
- Transmission infrastructure distance not included
- Land ownership (private vs federal vs state) not differentiated
- Next: add slope analysis from USGS DEM tiles, weight at 40%